# Laboratorio 1

## Integrantes

| Nombre                            | Carnet | Usuario Git |
| --------------------------------- | ------ | ----------- |
| Edwin Jose Gabriel De Leon Garcia | 22809  | EJGDLG      |
| Gustavo Adolfo Cruz Bardales      | 22779  | G2309       |
| Josué Emanuel Say Garcia          | 22801  | JosueSay    |
| Mathew Alexander Cordero Aquino   | 22982  | donmatthiuz |


## Repositorio

[Link al Repositorio](https://github.com/donmatthiuz/RL/tree/lab1)


## Task 1

Dado el contexto de la empresa de logística, diseñen formalmente el MDP que modela el problema. Su diseño debe especificar con precisión y justificación:

### Inciso 1

El espacio de estados **S**: ¿qué información debe contener el estado para que la propiedad de Markov se satisfaga razonablemente? Justifiquen cada variable que incluyen y cada variable que deciden omitir. Si omiten algo que podría ser relevante, expliquen qué consecuencia tiene esa omisión sobre la validez del modelo.

**Respuesta:**

La pregunta que gobierna el diseño del estado no es qué información existe, sino qué información es **suficiente** para que la propiedad de Markov se aproxime razonablemente. La propiedad no es una característica del mundo, sino de la representación: si falla, el estado está mal definido, no el entorno.

Propuesta:

$$
s = (x, y, b, \mathbf{d}, c, w)
$$

**Variables incluidas:**

| Variable | Dominio | Justificación |
|---|---|---|
| $(x, y)$ | $\{0,\dots,4\}^2$ | Posición del drone en la cuadrícula. Sin ella no hay noción de transición espacial. |
| $b$ | $\{0, 1, \dots, B\}$ | Nivel de batería discretizado. Determina qué acciones son viables y la proximidad al fallo. |
| $\mathbf{d}$ | $\{0,1\}^{k}$ | Vector binario de entregas pendientes sobre los $k$ puntos designados. Codifica el progreso de la misión. |
| $c$ | $\{0, 1\}$ | Indicador de carga a bordo (paquete cargado o no). Distingue "voy a entregar" de "voy de regreso". |
| $w$ | $\{0, 1, 2\}$ | Condición de viento o clima en tres niveles (nulo, moderado, adverso). Afecta la estocasticidad del movimiento y el consumo. |

La cardinalidad resultante es $|\mathcal{S}| = 25 \cdot (B{+}1) \cdot 2^{k} \cdot 2 \cdot 3$. Con $B = 10$ y $k = 3$ se obtienen $25 \cdot 11 \cdot 8 \cdot 2 \cdot 3 = 13{,}200$ estados: un espacio tabular perfectamente manejable para evaluación exacta de políticas.

El punto crítico es $\mathbf{d}$. Sin el vector de entregas pendientes, dos visitas a la misma coordenada en momentos distintos de la misión serían indistinguibles pese a tener valores completamente diferentes.

**Variables omitidas y sus consecuencias:**

- **Historial de trayectoria.** La posición actual más el vector de entregas ya resumen el pasado relevante para decidir el siguiente movimiento. Conservarla haría explotar $|\mathcal{S}|$ sin ganancia informativa.
- **Tiempo transcurrido absoluto.** El costo temporal se internaliza vía recompensa negativa por paso, sin necesidad de estado adicional. **Consecuencia:** el modelo no puede representar ventanas de entrega con hora límite; si el negocio las exige, habría que añadir un componente temporal y el espacio crecería linealmente en el número de intervalos.
- **Posición de otros drones.** Si los drones comparten espacio aéreo, el entorno se vuelve no estacionario desde la perspectiva de cada agente individual y la propiedad de Markov se rompe. Es la limitación más severa del diseño y se aborda en la sección de conclusiones.
- **Tráfico aéreo o zonas restringidas dinámicas.** Si las restricciones cambian durante el episodio, el agente las percibirá como ruido inexplicable.
- **Peso o tipo específico del paquete.** Si el consumo de batería dependiera fuertemente del peso, habría que incorporarlo a $s$ o el consumo real diferiría sistemáticamente del modelado.

Regla general: se incluye una variable si su omisión hace que dos situaciones con valores esperados distintos colapsen en el mismo estado.

### Inciso 2

El espacio de acciones **A**: definan las acciones disponibles para el drone. ¿Es discreto o continuo? ¿Hay acciones que deberían restringirse en ciertos estados? ¿Cómo modelan esa restricción dentro del MDP?

**Respuesta:**

El espacio de acciones es discreto y finito, con siete elementos:

$$
\mathcal{A} = \{\text{Norte},\ \text{Sur},\ \text{Este},\ \text{Oeste},\ \text{Entregar},\ \text{Recargar},\ \text{Esperar}\}
$$

- Las cuatro acciones de movimiento desplazan el drone una casilla.
- **Entregar** libera el paquete si el drone está sobre un punto de entrega pendiente con $c = 1$.
- **Recargar** restaura la batería, disponible únicamente en la base.
- **Esperar** mantiene la posición; es racional bajo viento adverso, cuando moverse tiene alta probabilidad de desvío.

Se elige discreto porque la cuadrícula ya impone una discretización natural del espacio, y un espacio de acciones finito permite evaluación tabular exacta, que es precisamente lo que la empresa quiere validar antes de invertir en aproximación funcional.

No todas las acciones son válidas en todo estado. Se define $\mathcal{A}(s) \subseteq \mathcal{A}$ como el conjunto de acciones admisibles. Existen dos formas de modelarlo dentro del MDP:

1. **Restricción estructural.** Definir $\pi(a \mid s) = 0$ para toda $a \notin \mathcal{A}(s)$ y normalizar sobre las acciones válidas; el agente nunca considera lo imposible.
2. **Restricción por transición y penalización.** Mantener $\mathcal{A}$ completo y hacer que las acciones inválidas transicionen al mismo estado con recompensa fuertemente negativa. El agente aprende la restricción en lugar de recibirla.

### Inciso 3

La función de recompensa **r(s, a, s')**: diseñen una función de recompensa que capture el objetivo real de la empresa. Consideren al menos tres objetivos potencialmente conflictivos: eficiencia de entrega, consumo de batería y seguridad de vuelo. ¿Cómo ponderan esos objetivos? ¿Qué consecuencias tendría una ponderación incorrecta sobre el comportamiento del agente?

**Respuesta:**

El agente optimiza lo que se le pidió, no lo que se pensó que se pidió; si la recompensa no captura el objetivo real, el agente encontrará formas de maximizarla que no anticipamos, y eso es responsabilidad de quien diseña, no del algoritmo.

Se plantean tres objetivos en tensión: **eficiencia de entrega**, **consumo de batería** y **seguridad de vuelo**. La función propuesta es aditiva ponderada:

$$
r(s,a,s') = w_{e} \cdot r_{\text{entrega}} + w_{t} \cdot r_{\text{tiempo}} + w_{b} \cdot r_{\text{bateria}} + w_{s} \cdot r_{\text{seguridad}}
$$

**Componentes y valores propuestos**

| Componente | Evento | Valor base | Peso |
|---|---|---|---|
| $r_{\text{entrega}}$ | Entrega completada con éxito | $+50$ | $w_e = 1.0$ |
| $r_{\text{entrega}}$ | Misión completa (todas las entregas + retorno a base) | $+100$ | $w_e = 1.0$ |
| $r_{\text{tiempo}}$ | Cada paso no terminal | $-1$ | $w_t = 1.0$ |
| $r_{\text{bateria}}$ | Consumo por movimiento | $-0.5$ por unidad | $w_b = 1.0$ |
| $r_{\text{bateria}}$ | Batería agotada en vuelo (pérdida del drone) | $-200$ | $w_b = 1.0$ |
| $r_{\text{seguridad}}$ | Colisión con obstáculo o edificio | $-500$ | $w_s = 1.0$ |
| $r_{\text{seguridad}}$ | Vuelo con $w = 2$ (viento adverso) | $-5$ | $w_s = 1.0$ |

Los valores están escalados de modo que el costo de un fallo catastrófico domine por un orden de magnitud sobre cualquier ganancia acumulable por eficiencia. Una entrega vale $+50$; una colisión vale $-500$. Un agente racional nunca cambiará diez entregas por un accidente.

**Consecuencias de una ponderación incorrecta**

La ponderación no es un detalle de ajuste, es la especificación del objetivo del negocio:

- **Sobrepeso de eficiencia ($w_t$ o $w_e$ demasiado alto).** El drone toma rutas directas sobre zonas peligrosas y vuela con batería crítica para ganar unos pasos. Optimiza la métrica de entrega y destruye la flota.
- **Sobrepeso de batería ($w_b$ demasiado alto).** El drone prefiere quedarse en la base recargando indefinidamente antes que arriesgar consumo. Si $|r_{\text{bateria}}|$ por movimiento supera el retorno descontado de una entrega, la política óptima es literalmente no despegar.
- **Sobrepeso de seguridad ($w_s$ demasiado alto).** El agente adopta una política ultraconservadora: espera indefinidamente cuando hay viento y solo se mueve en condiciones ideales. Cero accidentes, cero entregas.
- **Bonificación mal especificada.** Si se premia "acercarse al punto de entrega" en lugar de "entregar", el agente puede aprender a oscilar cerca del objetivo acumulando recompensa sin completar nunca la tarea. Es el caso clásico de *reward hacking*.

### Inciso 4

La función de transición **p(s' | s, a)**: ¿es determinista o estocástica? Si es estocástica, ¿qué fuentes de aleatoriedad existen en el dominio real y cómo las modelan? Escriban al menos tres transiciones concretas con sus probabilidades y justifiquen cada valor.

**Respuesta:**

Produciría un agente que planifica rutas óptimas bajo un supuesto de control perfecto y falla sistemáticamente en operación real.

**Fuentes de aleatoriedad en el dominio**

1. **Viento y turbulencia urbana.** Los cañones entre edificios generan ráfagas que desvían el drone de la trayectoria comandada.
2. **Error de actuación y sensado.** Los motores y el GPS tienen tolerancia; el desplazamiento real no coincide exactamente con el comandado.
3. **Fallo de entrega.** El punto de entrega puede estar obstruido o el receptor ausente, dejando el paquete a bordo.
4. **Consumo variable de batería.** Depende de la carga instantánea del motor, que varía con las condiciones.
5. **Cambio de condición climática.** $w$ evoluciona durante el episodio según su propia dinámica.

**Transiciones concretas**

**Transición 1 — Movimiento bajo viento nulo ($w = 0$), acción Norte desde $(2,2)$:**

$$
p\big((2,1) \mid (2,2), \text{Norte}\big) = 0.95, \quad
p\big((1,2) \mid \cdot \big) = 0.025, \quad
p\big((3,2) \mid \cdot \big) = 0.025
$$

El $5\%$ residual reparte simétricamente el error de actuación entre las dos casillas laterales, reflejando que el desvío no tiene dirección preferente en ausencia de viento. No se asigna probabilidad al retroceso porque un drone no se desplaza en sentido contrario al comando.

**Transición 2 — Movimiento bajo viento adverso ($w = 2$), acción Norte desde $(2,2)$:**

$$
p\big((2,1) \mid \cdot \big) = 0.70, \quad
p\big((1,2) \mid \cdot \big) = 0.15, \quad
p\big((3,2) \mid \cdot \big) = 0.10, \quad
p\big((2,2) \mid \cdot \big) = 0.05
$$

Bajo viento adverso la fiabilidad cae a $0.70$. La asimetría entre las laterales ($0.15$ frente a $0.10$) modela una dirección dominante del viento, información que es observable y por tanto legítima de incorporar. El $5\%$ de permanencia representa el caso en que la ráfaga cancela el avance. Estos valores son calibrables con telemetría real: la estructura del modelo es la contribución, los números concretos son una hipótesis inicial.

**Transición 3 — Acción Entregar sobre un punto designado con $c = 1$:**

$$
p\big(\mathbf{d}' = \mathbf{d} \setminus \{i\},\ c' = 0 \mid s, \text{Entregar}\big) = 0.90, \quad
p\big(\mathbf{d}' = \mathbf{d},\ c' = 1 \mid s, \text{Entregar}\big) = 0.10
$$

Un $10\%$ de fallo de entrega recoge las causas operativas habituales: zona de aterrizaje obstruida, receptor ausente, rechazo del paquete. La consecuencia de modelarlo es que el agente aprende a considerar reintentos en su planificación en lugar de asumir que una visita equivale a una entrega.

**Transición 4 — Consumo de batería (aplicable a toda acción de movimiento):**

$$
p(b' = b - 1 \mid \cdot) = 0.80, \quad p(b' = b - 2 \mid \cdot) = 0.20
$$

El consumo no es constante. Ese $20\%$ de consumo doble es lo que induce al agente a mantener un margen de seguridad en lugar de planificar al límite exacto de autonomía. Un modelo de consumo determinista produciría un agente que se queda sin batería con regularidad.

Todas las distribuciones cumplen $\sum_{s'} p(s' \mid s,a) = 1$ y, por construcción, dependen solo de $(s,a)$: la propiedad de Markov se preserva.


### Inciso 5

El factor de descuento **γ**: propongan un valor y justifíquenlo en términos del problema, no solo en términos matemáticos.

**Respuesta:**

Se propone $\gamma = 0.95$.

El horizonte efectivo de un factor de descuento es aproximadamente $1/(1-\gamma)$, lo que con $\gamma = 0.95$ da unos **20 pasos**. En una cuadrícula de $5 \times 5$, una misión típica de tres entregas con retorno a base ocupa entre 15 y 25 pasos. El agente valora hoy la recompensa de completar el recorrido y regresar, en vez de descartarla por lejana.

Los extremos ilustran por qué el valor importa:

- **$\gamma$ bajo (por ejemplo $0.5$, horizonte $\approx 2$ pasos).** El agente se vuelve miope. Toma la entrega más cercana, ignora que quedarse sin batería a mitad de ruta cuesta $-200$ y nunca aprende que regresar a la base tiene valor. Optimiza el siguiente movimiento, no la misión.
- **$\gamma = 1$ (sin descuento).** Es defendible porque el episodio es finito y termina en un estado terminal, igual que en el GridWorld canónico. Sin embargo, si existe cualquier probabilidad de que el episodio no termine (un drone que orbita indefinidamente sin agotar batería), la suma de retornos diverge y la evaluación de políticas no converge. $\gamma = 0.95$ garantiza convergencia sin sacrificar la visión de misión completa.
- **$\gamma$ muy alto ($0.999$, horizonte $\approx 1000$).** Innecesario. Introduce lentitud de convergencia sin aportar información relevante en un problema cuyo episodio dura decenas de pasos.

Hay además una lectura de negocio: $\gamma$ codifica cuánto vale para la empresa una entrega futura frente a una inmediata. Un $\gamma$ de $0.95$ dice que sí importa terminar la ruta completa, pero que la demora tiene un costo real.

**Diagrama MDP**

```mermaid
flowchart LR
    S["Estado s = (x, y, b, d, c, w)"] --> PI["Politica pi(a|s)"]
    PI --> A["Accion en A(s)"]
    A --> P["Transicion estocastica p"]
    S --> P
    P --> SP["Estado siguiente s prima"]
    P --> R["Recompensa ponderada r"]
    R --> V["Retorno descontado con gamma = 0.95"]
    SP --> S
```

## Task 2

Respondan las siguientes preguntas con argumentación técnica

### Inciso 1

Identifiquen al menos dos situaciones en las que la propiedad de Markov se violaría con su representación de estado actual. Para cada una, propongan una extensión del estado que restaure la propiedad y discutan el costo computacional de esa extensión.


**Respuesta:**

#### Primera Situación: Varios drones comparten el mismo espacio aereo.

Si un dron que lo llamaremos 1 llega al mismo estado $s=(2,2,,b,,\mathbf{d},,c,,w)$. En el primer caso, un dron 2 está a una casilla de distancia y convergiendo hacia la misma celda; en el segundo, 2 está en la esquina opuesta del grid.

El estado de 1 es identico a ambos casos pero $P(s' \mid s, a)$ es radicalmente distinto: En el primer caso hay riesgo que se choquen y la acción optima es desviarse. En el segundo ese riesgo no existe, por lo tanto se puede decir lo siguiente = "Dos estados idénticos con futuros condicionales distintos = violación directa de Markov"

#### Segunda Situación: Ventanas de entrega con hora límite

Supongamos que el punto de entrega $i$ tiene un plazo: debe completarse antes del paso 15. Dos episodios llegan al mismo $s=(x,y,b,\mathbf{d},c,w)$ con $i$ aún pendiente, uno en el paso 5 y otro en el paso 18. El estado observado es idéntico, pero en el primer caso todavía es viable cumplir el plazo y en el segundo ya no.



### Inciso 2

En el dominio de logística urbana, ¿qué tan razonable es el supuesto de que el agente puede observar el estado completo? ¿Qué variables del estado real probablemente no son directamente observables por el drone? ¿Cómo afecta eso la validez del modelo MDP versus un modelo POMDP?

**Respuesta:**


El supuesto de observabilidad completa es la pieza más frágil del modelo MDP tal como se planteó en la Tarea 1. Un MDP asume que el agente conoce $s=(x,y,b,\mathbf{d},c,w)$ sin ruido y sin retraso en cada paso; en un dron real eso es, en el mejor de los casos, una aproximación, y en el peor, una ficción cómoda para poder usar programación dinámica tabular.

En pocas palabras de las seis variables del estado, solo $\mathbf{d}$ y (con reservas) $c$ son observables sin ruido relevante. $(x,y)$ y $b$ son observables con error acotado y tratable. $w$ es la que más se aleja de "observable": el dron infiere una condición global a partir de una medición local.

Debido a estas limitaciones, el modelo MDP pierde validez como representación fiel del problema , ya que no se cumple el supuesto de estado completamente observable.Debido a esto, el problema se ajusta mejor a un modelo POMDP, donde el agente debe tomar decisiones basándose en observaciones parciales y mantener una creencia sobre el estado real del sistema. El uso de un MDP en este contexto es, por tanto, colocar de manera sencilla desde el punto de vista computacional, pero que introduce una diferencia notable entre el modelo y la realidad.

### Inciso 3

Argumenten si este problema debería modelarse como una tarea episódica o continua. ¿Cómo cambia esa decisión el diseño de la función de recompensa y el valor de γ?

**Respuesta:**

Es episodica esto porque: 
El propio diseño de la Tarea 1 ya asume esto implícitamente:
- La recompensa (Inciso 3) tiene eventos que solo tienen sentido como cierre de episodio: "misión completa" ($+100$) y "batería agotada en vuelo / pérdida del dron" ($-200$). Ambos son bonificaciones/penalizaciones que ocurren una vez y presuponen un estado terminal absorbente.
- La justificación de $\gamma$ (Inciso 5) ya lo dice explícitamente: "$\gamma=1$ es defendible porque el episodio es finito y termina en un estado terminal, igual que el GridWorld canónico."


Si el problema no fuera episódico sino continuo, sería necesario utilizar un valor de γ menor que 1 para garantizar la convergencia del retorno esperado y evitar que la suma de recompensas se vuelva infinita. En este tipo de entornos, no se puede depender de recompensas terminales únicas, ya que no existe un estado final definido; en su lugar, la función de recompensa debe diseñarse con señales más locales y recurrentes que guíen el comportamiento del agente a lo largo del tiempo.


## Task 3

Implementen en Python un evaluador iterativo de políticas para un GridWorld 5x5 que represente la ciudad de drones. Su implementación debe:

### Inciso 1

Representar el MDP completo: estados, acciones, función de transición y función de recompensa, consistentes con el diseño de la Tarea 1. No usen librerías de RL como Gymnasium. El MDP debe estar implementado desde cero.

### Inciso 2

Implementar la evaluación iterativa de políticas mediante aplicación repetida del operador de Bellman hasta convergencia. El criterio de convergencia debe ser configurable: el algoritmo termina cuando el cambio máximo en cualquier valor de estado entre dos iteraciones consecutivas es menor que un umbral θ.

### Inciso 3

Evaluar y comparar al menos tres políticas distintas: la política uniforme aleatoria, una política determinista fija definida por ustedes, y la política greedy derivada de los valores de la política anterior.

### Inciso 4

Producir para cada política evaluada: una tabla de valores (V^\pi(s)) para todos los estados, el número de iteraciones hasta convergencia, y una visualización del grid con los valores codificados en color y las acciones de la política greedy superpuestas como flechas.


## Task 4

Con base en los resultados de la implementación, redacten un dictamen técnico dirigido a la gerencia de la empresa de logística. El dictamen debe:

### Inciso 1

Comparar cuantitativamente las tres políticas evaluadas. ¿Cuál produce los valores más altos? ¿En qué estados difieren más? ¿La política greedy derivada de la política aleatoria es mejor que la política determinista que diseñaron manualmente?

### Inciso 2

Analizar la sensibilidad del resultado al valor de γ que propusieron en la Tarea 1. Repitan la evaluación con al menos dos valores adicionales de γ y discutan cómo cambian los valores y la política greedy resultante.

### Inciso 3

Identificar las limitaciones del modelo que implementaron respecto al problema real. ¿Qué aspectos del dominio de logística urbana no puede capturar este MDP? ¿Qué extensiones serían necesarias para que el modelo sea operacionalmente útil?

### Inciso 4

Concluir con una recomendación concreta: ¿es el MDP que diseñaron una base suficiente para construir un sistema de RL real para esta empresa? ¿Qué pasos siguientes recomendarían antes de proceder con la implementación completa?


## Referencias

- **[An Introduction to Markov Decision Processes](https://www.cs.rice.edu/~vardi/dag01/givan1.pdf)**
  
  Tratamiento formal de la tupla, la propiedad de Markov y las condiciones de existencia de política óptima.

- **[Understanding the Markov Decision Process (MDP)](https://builtin.com/machine-learning/markov-decision-process)**

  Enfoque aplicado al modelado de problemas reales como MDP.

- **[Markov Decision Processes (MDPs) - Structuring a Reinforcement Learning Problem](https://www.youtube.com/watch?v=my207WNoeyA)**

  Criterios prácticos para estructurar estado, acción y recompensa en un problema concreto.
